# Maximal Cliques for Tightening Set Packing Constraints: A case of MRI Magnet Design

**Team Members:** 
*  Vinicius Jameli 1010830255
*  Felix Deng 1005692476
*  Jie Mao 1003008121

**Table of Contents:** 
- [1. Introduction](#1-introduction)
- [2. Problem formulation](#2-problem-formulation)
- [3. Baseline model](#3-baseline-model)
- [4. Improvement: Sweep line maximal clique constraints](#4-improvement-sweep-line-maximal-clique-constraints)
- [5. Comparison of Experimental Results](#5-comparison-of-experimental-results)
- [6. Discussion and Propositions](#6-discussion-and-propositions)
- [7. References](#7-references)

In [ ]:
%pip install numpy cython gurobipy matplotlib tqdm requests

## 1. Introduction

Magnetic resonance imaging (MRI) has been one of the most promising modalities in soft-tissue imaging. The most important component of these machines are their electromagnets [1]. The medical images can only be generated by applying a strong and homogeneous magnetic field in a specific region in space that comes from such magnets [2]. Thus, the magnets designs are the most crucial step when manufacturing an MRI machine. Motivated by its importance, we study the optimal homogeneous magnet design problem.

Specifically, we design magnets for an open MRI machine with two coaxially located electromagnets positioned as shown in Figure 1 [3].
We extend the existing practices presented in [4] that models the problem as a mixed integer linear programming problem.
The goal is to design magnets such that their resultant magnetic field, given a fixed field magnitude, is homogeneous  (within a small tolerance), while minimizing the overall volume of the coils to minimize cost on the use of superconducting materials.  

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/mri_design.jpg?raw=true" width="50%">
    <p>Figure 1: Schematic representation of open coaxial MRI magnet design.</p>
</center>

## 2. Problem Formulation

In order to compute the resulting magnetic field generated by an electromagnetic coil in space, we first define a few variables, as illustrated in Figure 2: 

- Spatial coordinates: $(x, y, z)$
- Internal radius: $r$
- Cross section height: $h$
- Cross section width: $w$
- Number of wire windings: $N$
- Current: $i$

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/parameters.jpg?raw=true" width="60%">
    <p>Figure 2. Parameters for magnetic coil and spatial location in space.</p>
</center>

However, modelling the magnetic field generated by an ideal loop of coil based on its geometry involves highly nonlinear relationship. Instead, we exploit the fact that the current $i$ has a linear relationship with the resultant magnetic field. 
Therefore, we only consider $i$ as a variable and fix all other variables, and we are hence solve a large-scale mix-integer programming problem instead of solving a highly nonlinear but small optimization problem. 

In this project, we set $N = 10^7$ and let $w = 1.4 h$. We can further reduce the number of variables by defining $\sigma$ as the area of the cross-section, where $\sigma = h \times w$. Then, we discretize all the spatial coordinates $(x, y, z)$, widths $w$, and internal radius $r$. Such that, we can pre-compute an incidence matrix $\mathbf{A}$ as an input to the optimization model, where $a_{mn}^s$ is the magnetic field intensity generated by a unit current at the $m$-th target point in imaging domain $D$ generated by the $n$-th coil in the coil domain $\mathcal{N}$ with the $s$-th cross section in a finite set $\mathcal{S}$ that indexes all potential cross-sections that coils may adopt [5], as illustrated in Figure 3. 

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/domain.jpg?raw=true" width="60%">
    <p>Figure 3: Illustration of the optimization space in a cross-section of the magnetic coil, reproduced from Ref. [4].</p>
</center>

At the same time, we also make the following assumptions that are commonly used in the field of MRI technology: 

- We assume that the magnetic field is propagated in free space, and the coils are circular with rectangular cross-sections and are coaxial (i.e., share the same $z$-axis). Therefore, it is sufficient to constraint the magnetic field in a semi-circle in the $y=0$ plane due to symmetry, as shown in Figure 3. 
- The resultant magnetic field generated by a coil is linearly proportional to the current in the coil (i.e., linearity), and the resultant magnetic field generated by $N$ coils is given by the superposition of the magnetic fields generated by each of the $N$ coils. This assumption allows for a linear optimization model. 
- It is only necessary to constraint the axial component of the magnetic field ($B_z$) in order to guarantee homogeneity in the VOI, since the radial component ($B_r$ or $B_x$) becomes negligible through quadrature suppression when $B_z$ is homogeneous [3]. Therefore, we only need to constraint $M$ target points surrounding the surface boundary ($\partial D$) of the spherical domain $D$ (i.e., a set of target points $\mathcal{M}$). 
- In reality, the thickness of electrical coils is non-negligible, and we need to avoid overlapping coils in space. Let $\sigma^s$, $w^s$, and $h^s$ denote the area, width, and height of cross-section $s \in \mathcal{S}$. For each pair of candidate coil positions $n$ and $n'$, we define $\mathcal{S}_{nn'}$ as the set of overlapping cross-sections at these two coil locations. Cross-sections $s$ and $s'$ are considered overlapping at coil positions $n$ and $n'$ when $|{z_n - z_{n'}}| < \frac{h^s}{2} + \frac{h^{s'}}{2} + L_z$ and $|{r_n - r_{n'}}| < \frac{w^s}{2} + \frac{w^{s'}}{2} + L_r$, where $L_z$ and $L_r$ are the minimum required gaps between any two coils on the same pole along the $z$- and $r$-axis, respectively. 

Following the formulation originally developed by Dayarian et al. [4], the magnet design optimization problem can essentially be formulated as the following mix-integer linear programming (MILP) problem: 

$$
\begin{align}
\text{minimize} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} r_n \sigma^s x_{n}^{s} \\
\text{subject to} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \leq B_{0} (1 + \varepsilon), \quad \forall m \in \mathcal{M} \\
& \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \geq B_{0} (1 - \varepsilon), \quad \forall m \in \mathcal{M} \\
& |i_{n}^{s}| \leq J_{c} \sigma^{s} x_{n}^{s}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& \sum\limits_{s \in \mathcal{S}} x_{n}^{s} \leq 1, \quad \forall n \in \mathcal{N} \\
& x_{n}^{s} + x_{n'}^{s'} \leq 1, \quad \forall n, n' \in \mathcal{N}, \forall (s, s') \in \mathcal{S}_{nn'} \\
& i_{n}^{s} \text{ free}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& x_{n}^{s} \in \{0,1\}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} 
\end{align}
$$

Note that: 
- $i_n^s$ is a decision variable that takes the current associated with a thick coil at location $n$ in cross-section $s$. $i_n^s$ is used as a free variable in Eqn. 7, because it can take negative values for current passing through the reversed direction. 
- $x_n^s$ is a binary variable that takes the value of 1 if and only if coil $n$ in cross-section $s$ has a non-zero current, and it takes the value of 0 otherwise (Eqn. 8). 
- The presented MILP aims to minimize the total volume of superconductive material required for the magnet construction (Eqn. 1). 
- The constraints in Eqn. 2 and 3 enforce the magnetic field generated at each target point in the VOI is homogeneous with a magnetic intensity of $B_0$, where a small tolerance $\epsilon$ is allowed. 
- The constraint in Eqn. 4 ensures that the current density in active coils remains below the critical current density threshold $J_c$. 
- The constraint in Eqn. 5 ensures that each candidate coil location can take at most one cross-section.
- The constraint in Eqn. 6 prevents coil overlapping by prohibiting imcompatible cross-sections from being simultaneously assigned to any pair of coil positions $n$ and $n'$ according to the predifined set $\mathcal{S}_{nn'}$ (the last assumption discussed above). 

In [15]:
import numpy as np 
import time 

In [16]:
# Define constants for the problem 
B_0 = 0.5 # in Tesla, desired magnetic field strength 
EPSILON = 1e-5 # in Tesla, tolerance for magnetic field strength 
J_C = 2.1e8 # in A/m^2, critical current density 

In [17]:
# Setting the semi-circular imaging domain M on the x-z plane centered at origin 
VOI_RADIUS = 0.2 # in meters 
VOI_NUM_POINTS = 40 

angles = np.linspace(-np.pi / 2, np.pi / 2, VOI_NUM_POINTS) 
voi_boundary_points = np.zeros((VOI_NUM_POINTS, 3)) 
voi_boundary_points[:, 0] = VOI_RADIUS * np.cos(angles) # x-coordinates
voi_boundary_points[:, 1] = 0.0 # y-coordinates (all points lie on the x-z plane)
voi_boundary_points[:, 2] = VOI_RADIUS * np.sin(angles) # z-coordinates

In [18]:
# Coil domain N
COIL_NUM_WIDTHS = 20 # number of discrete widths to consider for the coil
COIL_WIDTH_MIN, COIL_WIDTH_MAX = 0.005, 0.10 # in meters
CROSS_SECTION_RATIO = 1.4 # w = h * ratio 

COIL_RADIUS_MIN, COIL_RADIUS_MAX = 0.10, 0.48 # in meters x20 --> 39
COIL_Z_MIN, COIL_Z_MAX = 0.34, 0.42 # in meters x10 --> 19
Z_AXIS_X, Z_AXIS_Y = 0.0, 0.0 # x and y coordinates of the z-axis where coils are placed

COIL_MIN_DISTANCE = 0.01 # in meters 
COIL_TURNS = 10000000 # number of wire turns 

# Compute the coil domain geometries 
def generate_coil_domain(coil_grid_spacing: float, generate_overlap_pairs: bool = True): 
    """
    Args:
        coil_grid_spacing (float): spacing between coil positions in meters
        generate_overlap_pairs (bool): whether to generate overlap pairs
    """

    coil_widths = np.linspace(COIL_WIDTH_MIN, COIL_WIDTH_MAX, COIL_NUM_WIDTHS)
    coil_heights = coil_widths / CROSS_SECTION_RATIO
    coil_cross_section_areas = coil_widths * coil_heights

    # Compute coil positions in a grid pattern 
    coil_z_positions = [] 
    z = COIL_Z_MIN 
    while z <= COIL_Z_MAX + 1e-10: 
        coil_z_positions.append(z) 
        z += coil_grid_spacing
    coil_radius_values = [] 
    r = COIL_RADIUS_MIN 
    while r <= COIL_RADIUS_MAX + 1e-10: 
        coil_radius_values.append(r) 
        r += coil_grid_spacing
        
    coil_positions = [] 
    coil_radii = []
    for z in coil_z_positions: 
        for r in coil_radius_values: 
            coil_positions.append([Z_AXIS_X, Z_AXIS_Y, z]) # all combinations for the north half 
            coil_radii.append(r)
    for z in coil_z_positions: 
        for r in coil_radius_values: 
            coil_positions.append([Z_AXIS_X, Z_AXIS_Y, -z]) # all combinations for the south half 
            coil_radii.append(r)
    coil_positions = np.array(coil_positions)
    coil_radii = np.array(coil_radii)
    
    num_coils = len(coil_positions)
    
    # Compute the S_nn' set to avoid coil overlapping 
    # Find all (n1, s1, n2, s2) pairs that would overlap 
    overlap_pairs = [] 

    def _would_overlap(n1: int, s1: int, n2: int, s2: int) -> bool:
        # Z overlap: centers within half-heights + min_distance
        z1 = coil_positions[n1, 2]
        z2 = coil_positions[n2, 2]
        h1 = coil_heights[s1]
        h2 = coil_heights[s2]
        z_overlap = abs(z1 - z2) < (h1 / 2 + h2 / 2 + COIL_MIN_DISTANCE)

        # Radius overlap: radii within half-widths + min_distance
        r1 = coil_radii[n1]
        r2 = coil_radii[n2]
        w1 = coil_widths[s1]
        w2 = coil_widths[s2]
        r_overlap = abs(r1 - r2) < (w1 / 2 + w2 / 2 + COIL_MIN_DISTANCE)

        return z_overlap and r_overlap
    
    if generate_overlap_pairs:
        st = time.time() 
        # North pole coils only: south pole overlap constraints are redundant given the symmetry constraint
        for n1 in range(num_coils // 2): 
            for n2 in range(n1 + 1, num_coils // 2):
                for s1 in range(COIL_NUM_WIDTHS):
                    for s2 in range(COIL_NUM_WIDTHS):
                        if _would_overlap(n1, s1, n2, s2):
                            overlap_pairs.append((n1, s1, n2, s2))
        print("Overlap pairs generated in {:.2f} seconds".format(time.time() - st))
                    
    return coil_widths, coil_heights, coil_cross_section_areas, coil_positions, coil_radii, overlap_pairs

In [19]:
# Define a data structure for the A_mn^s matrix in Eqn. (2) and (3) 

class AMNSMatrix:
    """Container for AMNS precomputed field matrices.

    The AMNS matrix is a 4D tensor with dimensions:
    - c: Field component (0=Bx, 1=By, 2=Bz)
    - m: Optimization point index
    - n: Coil position index
    - w: Width index

    Attributes:
        Bx: Field x-component matrix [num_opt_points, num_coils, num_widths]
        By: Field y-component matrix [num_opt_points, num_coils, num_widths]
        Bz: Field z-component matrix [num_opt_points, num_coils, num_widths]
        num_opt_points: Number of optimization points
        num_coils: Number of coil positions
        num_widths: Number of width values
    """

    Bx: np.ndarray
    By: np.ndarray
    Bz: np.ndarray
    num_opt_points: int
    num_coils: int
    num_widths: int
    
    def __init__(self, Bx: np.ndarray, By: np.ndarray, Bz: np.ndarray,
                 num_opt_points: int, num_coils: int, num_widths: int):
        self.Bx = Bx
        self.By = By
        self.Bz = Bz
        self.num_opt_points = num_opt_points
        self.num_coils = num_coils
        self.num_widths = num_widths

    @property
    def shape(self):
        """Return the shape of the AMNS tensor."""
        return (3, self.num_opt_points, self.num_coils, self.num_widths)

    def __getitem__(self, idx):
        """Allow indexing AMNS[c, m, n, w] or AMNS[c] for component."""
        if isinstance(idx, int):
            if idx == 0:
                return self.Bx
            elif idx == 1:
                return self.By
            elif idx == 2:
                return self.Bz
            else:
                raise IndexError(f"Component index must be 0, 1, or 2, got {idx}")
        elif isinstance(idx, tuple):
            c = idx[0]
            rest = idx[1:]
            if c == 0:
                return self.Bx[rest]
            elif c == 1:
                return self.By[rest]
            elif c == 2:
                return self.Bz[rest]
            else:
                raise IndexError(f"Component index must be 0, 1, or 2, got {c}")

In [20]:
# Since the AMNS matrix can be computationally expensive to compute, we will load the precomputed matrices from GitHub. 
# The detail method in generating these matrics can be found in this directory: https://github.com/Vjameli/magnet-design-MIE1603/blob/main/compute_amns.py 
import requests 
from io import BytesIO 

def load_amns(url: str) -> AMNSMatrix:
    """Load the precomputed AMNS matrix from our GitHub repository 

    Args:
        url (str): link to the GitHub file source 

    Returns:
        AMNSMatrix: the loaded AMNS matrix casted into our AMNSMatrix data structure
    """
    response = requests.get(url)
    response.raise_for_status()  # Check if the request was successful
    data = np.load(BytesIO(response.content))
    return AMNSMatrix(
        Bx=data["Bx"],
        By=data["By"],
        Bz=data["Bz"],
        num_opt_points=int(data["num_opt_points"]),
        num_coils=int(data["num_coils"]),
        num_widths=int(data["num_widths"]),
    )


## 3. Baseline Model

For the baseline, we implemented the MILP problem modelled with Eqn. 1 to 8 using Gurobi. 

In [21]:
import gurobipy as gp
from gurobipy import GRB


def solve_baseline_model(
    coil_positions: np.ndarray, coil_radii: np.ndarray, coil_cross_section_areas: np.ndarray, 
    amns_matrix: AMNSMatrix, overlap_pairs: list,
    time_limit: int = 600,
): 
    model_baseline = gp.Model("baseline_coil_optimization") 

    # Gurobi solver configurations 
    model_baseline.setParam('TimeLimit', time_limit) # seconds 
    model_baseline.setParam('MIPGap', 0.01) # 1% optimality gap
    model_baseline.setParam('FeasibilityTol', 1e-8)
    model_baseline.setParam('IntegralityFocus', 1) 
    model_baseline.setParam('Threads', 1) 
    model_baseline.setParam('OutputFlag', 1) # 0=silent, 1=normal
    model_baseline.setParam('Seed', 42)

    # Create decision variables 
    num_coils = len(coil_positions)
    num_opt_points = amns_matrix.num_opt_points  # derived from AMNS, not global VOI_NUM_POINTS

    # i[n, s]: Current through coil n with width s (continuous) [Eqn. 7]
    vars_i = model_baseline.addVars(
        num_coils,
        COIL_NUM_WIDTHS,
        lb=-GRB.INFINITY,
        ub=GRB.INFINITY,
        vtype=GRB.CONTINUOUS,
        name="i",
    )
    # x[n, s]: Binary indicator if coil n uses width s [Eqn. 8]
    vars_x = model_baseline.addVars(
        num_coils,
        COIL_NUM_WIDTHS,
        vtype=GRB.BINARY,
        name="x",
    )

    # Convert to 2D arrays for easier indexing
    array_i = np.array(
        [[vars_i[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(num_coils)],
        dtype=object,
    )
    array_x = np.array(
        [[vars_x[n, s] for s in range(COIL_NUM_WIDTHS)] for n in range(num_coils)],
        dtype=object,
    )

    # Set objective: minimize total "coil volume" (radius * area) [Eqn. 1]
    # Scaled by 1e4 for numerical stability (matches original experiment)
    OBJECTIVE_SCALE = 1e4
    obj = gp.LinExpr(
        (
            OBJECTIVE_SCALE * coil_radii[:, None] * coil_cross_section_areas[None, :]).flatten().tolist(), 
            list(array_x.flatten()
        )
    )
    model_baseline.setObjective(obj, GRB.MINIMIZE)
    
    # Add homogeneity constraints on Bz field [Eqn. 2 and 3]
    upper_bound = B_0 * (1 + EPSILON)
    lower_bound = B_0 * (1 - EPSILON)

    if amns_matrix.Bz.ndim == 3: 
        bz_2d = amns_matrix.Bz.reshape(num_opt_points, num_coils * COIL_NUM_WIDTHS)
    else: 
        bz_2d = amns_matrix.Bz # already (num_opt_points, num_coils * num_widths)
    i_flat = list(array_i.flatten())

    for m in range(num_opt_points):
        expr = gp.LinExpr(bz_2d[m].tolist(), i_flat)
        model_baseline.addConstr(expr >= lower_bound, name=f"homo_lower_{m}")
        model_baseline.addConstr(expr <= upper_bound, name=f"homo_upper_{m}")
    
    # Add symmetry constraint between north and south pole coils 
    half_num_coils = num_coils // 2
    for n in range(half_num_coils):
        for s in range(COIL_NUM_WIDTHS):
            model_baseline.addConstr(
                array_x[n, s] == array_x[n + half_num_coils, s],
                name=f"symmetry_{n}_{s}"
            )
    
    # Add upper bound constraint on current density [Eqn. 4]
    for n in range(num_coils): 
        for s in range(COIL_NUM_WIDTHS): 
            i_max = J_C * coil_cross_section_areas[s] / COIL_TURNS 
            model_baseline.addConstr(array_i[n, s] <= i_max * array_x[n, s], name=f"current_upper_{n}_{s}")
            model_baseline.addConstr(array_i[n, s] >= -i_max * array_x[n, s], name=f"current_lower_{n}_{s}")
            
    # Add constraint such that each coil position uses at most one width [Eqn. 5]
    for n in range(num_coils): 
        expr = gp.LinExpr() 
        for s in range(COIL_NUM_WIDTHS): 
            expr += array_x[n, s]
        model_baseline.addConstr(expr <= 1, name=f"unique_width_{n}")
        
    # Add constraint to prevent overlapping coils [Eqn. 6] (north pole only)
    for n1, s1, n2, s2 in overlap_pairs: 
        model_baseline.addConstr(
            array_x[n1, s1] + array_x[n2, s2] <= 1, 
            name=f"overlap_{n1}_{s1}_{n2}_{s2}"
        )
        if s1 != s2:
            model_baseline.addConstr(
                array_x[n1, s2] + array_x[n2, s1] <= 1,
                name=f"overlap_cross_{n1}_{s2}_{n2}_{s1}",
            )
            
    model_baseline.update()
    start_time = time.time()
    model_baseline.optimize() 
    solve_time = time.time() - start_time

    print("Solve time (seconds):", solve_time)
    print("Optimization status:", model_baseline.Status)
    print("Objective value:", model_baseline.ObjVal if model_baseline.SolCount > 0 else float("inf"))

### 3.1 Experiment with a smaller instance 

In [22]:
coil_widths, coil_heights, coil_cross_section_areas, coil_positions, coil_radii, overlap_pairs = generate_coil_domain(coil_grid_spacing=0.02)

print("Number of optimizing points:", VOI_NUM_POINTS)
print("Number of coil configurations:", len(coil_positions))
print("Number of cross-sectional area options:", COIL_NUM_WIDTHS)

Overlap pairs generated in 1.73 seconds
Number of optimizing points: 40
Number of coil configurations: 200
Number of cross-sectional area options: 20


In [23]:
AMNS_MATRIX_40_200_20 = load_amns("https://github.com/Vjameli/magnet-design-MIE1603/raw/refs/heads/main/AMNS_40_200_20.npz")
print("Loaded AMNS matrix with shape:", AMNS_MATRIX_40_200_20.Bx.shape)

Loaded AMNS matrix with shape: (40, 200, 20)


In [24]:
solve_baseline_model(
    coil_positions, coil_radii, coil_cross_section_areas, AMNS_MATRIX_40_200_20, overlap_pairs
)

Set parameter TimeLimit to value 600
Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 1e-08
Set parameter IntegralityFocus to value 1
Set parameter Threads to value 1
Set parameter OutputFlag to value 1
Set parameter Seed to value 42
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  600
FeasibilityTol  1e-08
MIPGap  0.01
IntegralityFocus  1
Seed  42
Threads  1

Optimize a model with 810772 rows, 8000 columns and 1944984 nonzeros (Min)
Model fingerprint: 0x19cad80f
Model has 4000 linear objective coefficients
Variable types: 4000 continuous, 4000 integer (4000 binary)
Coefficient statistics:
  Matrix range     [4e-04, 2e+01]
  Objective range  [2e-02, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-01, 1e+00]
Presolve removed 785755 rows and 2000 columns
Presolve time: 0.84s
Presolv

The run above based on a dataset with $|N|=200$ and $|S|=20$ would generate an optimal solution as visualized in Figure 4. Since generating this figure is highly resource-intensive, involving a process that is similar to the computation of the AMNS matrix above, we have left out the code implementation in this notebook and only included the figure produced. 

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/pairwise_plot.png?raw=true" width="60%">
    <p>Figure 4: Optimal solution of test case<p>
</center>

### 3.2 Experiment with a larger instance 

Next, decrease the step size used to create possible center point of coil instances to build a new dataset with $|N|=702$ and more optimization points (80 boundary + 315 interior = 395 total, constraining the field inside the volume of interest as well).

In [25]:
coil_widths, coil_heights, coil_cross_section_areas, coil_positions, coil_radii, overlap_pairs = generate_coil_domain(coil_grid_spacing=0.01)

print("Number of optimizing points: 395 (80 boundary + 315 interior)")
print("Number of coil configurations:", len(coil_positions))
print("Number of cross-sectional area options:", COIL_NUM_WIDTHS)

Overlap pairs generated in 20.22 seconds
Number of optimizing points: 395 (80 boundary + 315 interior)
Number of coil configurations: 702
Number of cross-sectional area options: 20


In [26]:
# Load the 395-point AMNS (80 boundary + 315 interior) to match the original experiment
AMNS_MATRIX_395_702_20 = load_amns("https://github.com/Vjameli/magnet-design-MIE1603/raw/refs/heads/main/AMNS_395_702_20.npz")
print("Loaded AMNS matrix with shape:", AMNS_MATRIX_395_702_20.Bz.shape)

Loaded AMNS matrix with shape: (395, 702, 20)


In [27]:
solve_baseline_model(
    coil_positions, coil_radii, coil_cross_section_areas, AMNS_MATRIX_395_702_20, overlap_pairs,
    time_limit=180,
)

Set parameter TimeLimit to value 180
Set parameter MIPGap to value 0.01
Set parameter FeasibilityTol to value 1e-08
Set parameter IntegralityFocus to value 1
Set parameter Threads to value 1
Set parameter OutputFlag to value 1
Set parameter Seed to value 42
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 1 threads

Non-default parameters:
TimeLimit  180
FeasibilityTol  1e-08
MIPGap  0.01
IntegralityFocus  1
Seed  42
Threads  1

Optimize a model with 10908570 rows, 28080 columns and 32919796 nonzeros (Min)
Model fingerprint: 0xf0a7c83a
Model has 14040 linear objective coefficients
Variable types: 14040 continuous, 14040 integer (14040 binary)
Coefficient statistics:
  Matrix range     [4e-04, 2e+01]
  Objective range  [2e-02, 3e+01]
  Bounds range     [1e+00, 1e+00]
  RHS range        [5e-01, 1e+00]
Presolve removed 0 rows and 0 columns (presolve time = 5s)...
Pres

**Note:** The time limit for this run is set to 180 seconds. This instance is numerically unstable — the LP relaxation at the root node will diverge on most computers, so the solver will not make meaningful progress beyond the root node within any reasonable time budget.

## 4. Improvement: Sweep Line Maximal Clique Constraints 

### 4.1 Alternative Formulation

To improve the baseline model, we reformulate the overlapping constraint, i.e. Eq (6) in the Baseline Formulation.

$$
x_{n}^{s} + x_{n'}^{s'} \leq 1, \quad \forall n, n' \in \mathcal{N}, \forall (s, s') \in \mathcal{S}_{nn'}
$$

This approach requires $\mathcal{O}((N \cdot S)^2)$ constraints in the worst case, which is prohibitively large - creating a massive memory bottleneck and weakening the linear programming (LP) relaxation of the problem.

The first step towards strengthening this formulation is defining its graph representation. Let $G=(\mathcal{V},\mathcal{E})$ be a **conflict graph** where each vertex $v \in \mathcal{V} = \mathcal{N} \times \mathcal{S} $ represents coil $(n,s)$, and an edge $e \in \mathcal{E}$ exists between two vertices if the corresponding coils overlap.
A **clique** $\mathcal{C} \subseteq V$ is a subset of vertices where every two distinct vertices are adjacent.
Consequently, at most one coil from any clique can be active simultaneously.

In the literature, whenever set packing constraints arise, the practice has been to create the conflict graph of these overlaps to extract cliques and tighten the formulation [6] - in particular, **maximal cliques**. A maximal clique is a clique that can not be extended by including one more adjacent vertex, i.e., it is not a subset of a larger clique. 

Furthermore, Padberg has proven [7] that maximal cliques of the conflict graph define a first set of facets for the set-packing polyhedron - that is, for the polyhedron defined by set packing constraints. He also shows that cycles without chords of odd length of the intersection graph give rise to a further set of facets.

In fact, the only reason why gurobi can actually solve the pairwise is becaue its pre-solver will generate the conflict graph by analyzing the constraints and compute cliques to reduce the number of constraints. However, there are two major drawbacks of entrusting Gurobi with this task: 

1. The cliques generated by Gurobi are far from being maximal. This means the formulation is still far away from tight, let alone being facet-defining. As a result, the number of constraints that remain is still significant.
2. Gurobi has to fit all the pairwise constraints in the RAM during pre-solving, causing the problem to reach its RAM bottleneck much quicker.

Therefore, a set-packing constraint from a clique $\mathcal{C}$ is simply defined as:

$$\sum_{(n,s) \in \mathcal{C}} x_n^s \leq 1 \tag{6a}$$

This constraint is significantly stronger than the pairwise formulation, as it cuts off the fractional solutions
(e.g., $x_i=x_j=x_k=0.5$) that satisfy pairwise constraints but violate the clique constraint (e.g., $x_i+x_j+x_k \le 1$).

We replace Eq (6) with the Eq (6a) above, giving us the alternative formulation.

$$
\begin{align}
\text{minimize} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} r_n \sigma^s x_{n}^{s} \\
\text{subject to} \quad & \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \leq B_{0} (1 + \varepsilon), \quad \forall m \in \mathcal{M} \\
& \sum\limits_{n \in \mathcal{N}} \sum\limits_{s \in \mathcal{S}} a_{mn}^{s} i_{n}^{s} \geq B_{0} (1 - \varepsilon), \quad \forall m \in \mathcal{M} \\
& |i_{n}^{s}| \leq J_{c} \sigma^{s} x_{n}^{s}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \\
& \sum_{(n,s) \in \mathcal{C}} x_n^s \leq 1 \tag{6a} \\
& i_{n}^{s} \text{ free}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \tag{7} \\
& x_{n}^{s} \in \{0,1\}, \quad \forall n \in \mathcal{N}, \forall s \in \mathcal{S} \tag{8}
\end{align}
$$



### 4.2 Sweepline Maximal Clique Algorithm

Since all the coils share the same axial axis (i.e., the $z$ axis), the coil overlap boils down to overlap of rectangles in the plane $y=0$. Crucially, a family of subsets of the set of rectangles $R_i$ $(i = 1,\dots,n)$ with sides parallel to the axes satisfies the Helly property. That is, for any subset of $(R_{i_1},\ldots,R_{i_k})$ of rectangles of which every pair intersects, all the rectangles contained in the subset have nonempty intersection [8]. In other words, two or more rectangles overlap if there is a region in space shared by them. Luckily, [8] have showed that there is a polynomial algorithm, $\mathcal{O}(n\log n)$ for finding the maximum clique in an intersection graph of rectangles. The proposed algorithm can be easily modified to enumerate all the maximal cliques in polynomial time by just examining the geometry of overlap, without ever building the conflict graph.

Imai \& Asano's algorithm [8] falls into the category of sweep line algorithms, commonly used for solving planar problems. Imagine sweeping a line over the plane $y=0$ checking for "events" - defined as rectangles overlapping at a certain point in space. Every time we hit an event, we compute the clique that happened at that point by enumerate the rectangles overlapping at that point, and move to the next event. If we locate the second edge of one of the overlapping rectangles, we have found a maximal clique.

Thus, our **methodological contribution** is a new way of modeling set-packing constraints that represents non-overlapping of rectangles in 2D. Our method exploits the geometry to derive all maximal cliques in polynomial time which provides a very tight formulation for the problem with orders of magnitude less constraints. These will be shown in the experimental runs in Section 4.3 and 4.4 below. Overall, this shows that instead of constructing the conflict graph, much insight can be extract directly from the geometry of overlap!

This approach can also be applied to other OR problems. For instance, in the graphical feature label placement problem for automated cartography [9], the optimization of punch press tool design [10] and in maritime logistics, where in the continuous berth allocation problem explicitly models vessels as 2D rectangles in a space-time graph to prevent spatial and temporal overlaps [11].

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/Algorithm_1.png?raw=true" width="80%">
</center>

Below is an implementation of the alternative formulation using the Sweep-line Maximal Clique Enumeration algorithm.

In [28]:
%load_ext Cython

In [29]:
%%cython -+

# ---------------------------------------------------------
# SWEEP-LINE MAXIMAL CLIQUE ALGORITHM in PYTHON
# ---------------------------------------------------------

# cython: language_level=3
# cython: boundscheck=False
# cython: wraparound=False
# cython: cdivision=True
"""Cython/C++ implementation of sweep line maximal clique enumeration.

Provides SweepLineCliqueEnumeratorCpp, a drop-in replacement for
SweepLineCliqueEnumerator that uses C++ STL data structures (std::set,
std::vector, std::unordered_map, std::sort, std::includes) for all
inner-loop operations, eliminating Python overhead in the hot paths.
"""

import time
import numpy as np
cimport numpy as np
np.import_array()

from libc.stdlib cimport malloc, free
from libcpp.vector cimport vector
from libcpp.pair cimport pair
from libcpp.set cimport set as cpp_set
from libcpp.unordered_map cimport unordered_map
from libcpp.algorithm cimport sort, includes
from cython.operator cimport dereference as deref, preincrement as inc

# Define SweepLineStats directly here
cdef class SweepLineStats:
    cdef public int num_raw_cliques
    cdef public int num_maximal_cliques
    cdef public int total_members
    cdef public float elapsed_sweep_s
    cdef public float elapsed_filter_s
    cdef public float elapsed_total_s

    def __init__(self, num_raw_cliques: int, num_maximal_cliques: int, total_members: int,
                 elapsed_sweep_s: float, elapsed_filter_s: float, elapsed_total_s: float):
        self.num_raw_cliques = num_raw_cliques
        self.num_maximal_cliques = num_maximal_cliques
        self.total_members = total_members
        self.elapsed_sweep_s = elapsed_sweep_s
        self.elapsed_filter_s = elapsed_filter_s
        self.elapsed_total_s = elapsed_total_s


# ---------------------------------------------------------------------------
# Type aliases
# ---------------------------------------------------------------------------
ctypedef pair[int, int] IntPair
ctypedef pair[double, IntPair] Event   # (coord, (etype, encoded_idx))
                                       # etype: 0=END, 1=START (END sorts first)
ctypedef vector[int] Clique            # sorted encoded indices: n * num_widths + s
ctypedef cpp_set[Clique] CliqueSet


# ---------------------------------------------------------------------------
# C-level helper functions
# ---------------------------------------------------------------------------

cdef inline bint _is_proper_subset(vector[int]& c, vector[int]& sup) nogil:
    """Return True iff sorted vector c is a proper subset of sorted vector sup."""
    if c.size() >= sup.size():
        return False
    return includes(sup.begin(), sup.end(), c.begin(), c.end())


cdef void _solve_1d_cliques(
    cpp_set[int]& active,
    double* r_lo,
    double* r_hi,
    CliqueSet& raw_cliques,
):
    """Sweep the r-axis over the active set and append maximal 1D cliques.

    All pairs in active already overlap in z; here we check r-overlap only.
    Cliques are inserted into raw_cliques (a sorted set, so insertion
    deduplicates automatically).

    Args:
        active:      Set of encoded indices currently active in the z-sweep.
        r_lo/r_hi:  C arrays of r lower/upper bounds indexed by encoded idx.
        raw_cliques: Output set; cliques are added in place.
    """
    cdef:
        vector[Event] r_events
        Event ev
        cpp_set[int] curr
        cpp_set[int].iterator c_it
        Clique clique_vec
        int idx, i

    # Build r-events from active set
    c_it = active.begin()
    while c_it != active.end():
        idx = deref(c_it)
        ev.first = r_lo[idx]
        ev.second.first = 1    # START
        ev.second.second = idx
        r_events.push_back(ev)
        ev.first = r_hi[idx]
        ev.second.first = 0    # END
        ev.second.second = idx
        r_events.push_back(ev)
        inc(c_it)

    sort(r_events.begin(), r_events.end())

    curr.clear()
    for i in range(r_events.size()):
        ev = r_events[i]
        if ev.second.first == 1:  # START
            curr.insert(ev.second.second)
        else:                      # END — emit current set before removing
            if curr.size() >= 2:
                clique_vec.clear()
                c_it = curr.begin()
                while c_it != curr.end():
                    clique_vec.push_back(deref(c_it))
                    inc(c_it)
                # cpp_set<int> is ordered, so clique_vec is already sorted
                raw_cliques.insert(clique_vec)
            curr.erase(ev.second.second)


cdef object _clique_to_frozenset(vector[int]& c, int num_widths):
    """Decode an encoded Clique back to a Python frozenset of (n, s) tuples."""
    cdef list pairs = []
    cdef int j, enc
    for j in range(c.size()):
        enc = c[j]
        pairs.append((enc // num_widths, enc % num_widths))
    return frozenset(pairs)


cdef vector[Clique] _filter_subsets_cpp(CliqueSet& raw_cliques_set):
    """Remove proper subsets using inverted-index + pivot strategy.

    Pure C++ version: clears raw_cliques_set immediately after copying to halve
    peak C++ memory, and returns survivors as C++ vectors so no Python objects
    are created during intermediate flushes.

    Args:
        raw_cliques_set: Set of candidate cliques. CLEARED by this function
                         right after copying, to free the set's node memory.

    Returns:
        Vector of maximal cliques as sorted int vectors (no Python decoding).
    """
    cdef:
        CliqueSet.iterator set_it
        vector[Clique] candidates
        vector[int] accepted_indices
        unordered_map[int, vector[int]] index_map
        unordered_map[int, vector[int]].iterator map_it
        vector[int] pivot_list
        bint is_new, is_sub
        int i, j, k, key, min_len, min_key, acc_idx, cur_len, sup_i, ci_size
        vector[IntPair] size_idx   # (size, original_index) pairs for sorting
        IntPair p
        vector[Clique] result

    # 1. Copy all cliques from the set into a vector for random access.
    set_it = raw_cliques_set.begin()
    while set_it != raw_cliques_set.end():
        candidates.push_back(deref(set_it))
        inc(set_it)

    # 2. Clear the set immediately: frees every Red-Black tree node, cutting
    #    C++ peak memory roughly in half before the filter work begins.
    raw_cliques_set.clear()

    # 3. Sort indices by descending size using C++ pairs (no Python objects).
    #    IntPair.first = size, IntPair.second = original index.
    #    std::sort on pairs sorts ascending; iterating backwards gives largest first.
    for i in range(<int>candidates.size()):
        p.first = <int>candidates[i].size()
        p.second = i
        size_idx.push_back(p)
    sort(size_idx.begin(), size_idx.end())

    # 4. Inverted-index filter (process largest cliques first).
    for k in range(<int>size_idx.size() - 1, -1, -1):
        i = size_idx[k].second
        ci_size = size_idx[k].first

        min_len = 2147483647
        min_key = -1
        is_new = False

        for j in range(ci_size):
            key = candidates[i][j]
            map_it = index_map.find(key)
            if map_it == index_map.end():
                is_new = True
                break
            cur_len = <int>deref(map_it).second.size()
            if cur_len < min_len:
                min_len = cur_len
                min_key = key

        if is_new:
            acc_idx = <int>accepted_indices.size()
            accepted_indices.push_back(i)
            for j in range(ci_size):
                index_map[candidates[i][j]].push_back(acc_idx)
            continue

        is_sub = False
        if min_key >= 0:
            pivot_list = index_map[min_key]
            for j in range(<int>pivot_list.size()):
                sup_i = accepted_indices[pivot_list[j]]
                if _is_proper_subset(candidates[i], candidates[sup_i]):
                    is_sub = True
                    break

        if not is_sub:
            acc_idx = <int>accepted_indices.size()
            accepted_indices.push_back(i)
            for j in range(ci_size):
                index_map[candidates[i][j]].push_back(acc_idx)

    # 5. Collect accepted cliques into result (still C++ vectors, no Python).
    for j in range(<int>accepted_indices.size()):
        result.push_back(candidates[accepted_indices[j]])

    return result


# ---------------------------------------------------------------------------
# Public extension type
# ---------------------------------------------------------------------------

cdef class SweepLineCliqueEnumeratorCpp:
    """C++ backed sweep line clique enumerator.

    Drop-in replacement for SweepLineCliqueEnumerator. Uses C++ STL containers
    for all inner-loop operations, eliminating Python overhead in the hot paths
    (event building, active-set management, deduplication, subset filtering).

    Attributes:
        positions:     Coil center positions, shape (num_coils, 3). Column 2 is z.
        radii:         Coil radii, shape (num_coils,).
        widths:        Cross-section widths, shape (num_widths,).
        heights:       Cross-section heights, shape (num_widths,).
        min_distance:  Minimum required separation between coils (L parameter).
        last_run_stats: SweepLineStats from the last enumerate_cliques() call
                        (None before first run).
    """

    cdef public object positions
    cdef public object radii
    cdef public object widths
    cdef public object heights
    cdef public double min_distance
    cdef public object last_run_stats

    def __init__(
        self,
        positions,
        radii,
        widths,
        heights,
        double min_distance,
    ):
        self.positions = np.asarray(positions, dtype=np.float64)
        self.radii = np.asarray(radii, dtype=np.float64)
        self.widths = np.asarray(widths, dtype=np.float64)
        self.heights = np.asarray(heights, dtype=np.float64)
        self.min_distance = min_distance
        self.last_run_stats = None

    def enumerate_cliques(self, bint north_only=True, int max_raw_cliques=1000000):
        """Run the C++ sweep-line algorithm and return all maximal cliques.

        Args:
            north_only: If True (default), enumerate cliques only for the north
                pole coils (indices 0 .. num_coils//2 - 1).
            max_raw_cliques: Cap on raw clique count that triggers an intermediate
                flush. When the sweep generates more than this many raw cliques,
                _filter_subsets is run immediately and the survivors are
                re-inserted so the sweep continues with bounded memory.
                Default: 1_000_000.

        Returns:
            List of frozensets, each containing (coil_idx, width_idx) pairs
            that form a maximal clique in the conflict graph.
        """
        cdef:
            double[:, ::1] pos
            double[::1] radii_mv
            double[::1] widths_mv
            double[::1] heights_mv
            int num_coils, n_north, num_widths, n_vars
            int n, s, encoded, i, j, num_events
            double z_n, r_n, h_s, w_s, half_L
            double* z_lo_arr
            double* z_hi_arr
            double* r_lo_arr
            double* r_hi_arr
            vector[Event] z_events
            Event ev
            cpp_set[int] active
            CliqueSet raw_cliques
            double elapsed_sweep, elapsed, elapsed_filter
            int n_raw
            list maximal
            vector[Clique] survivors   # pure C++ survivor storage (no Python objects)
            int survivor_count         # survivors after each flush
            int n_intermediate_filters # counts how many flushes have occurred
            int _last_print_raw        # tracks raw count at last progress print

        t_start = time.perf_counter()
        print(
            f"[SweepLineCpp] enumerate_cliques: north_only={north_only}, "
            f"max_raw_cliques={max_raw_cliques}",
            flush=True,
        )

        pos = np.ascontiguousarray(self.positions, dtype=np.float64)
        radii_mv = np.ascontiguousarray(self.radii, dtype=np.float64)
        widths_mv = np.ascontiguousarray(self.widths, dtype=np.float64)
        heights_mv = np.ascontiguousarray(self.heights, dtype=np.float64)

        num_coils = pos.shape[0]
        num_widths = widths_mv.shape[0]

        n_north = num_coils // 2 if north_only else num_coils
        n_vars = n_north * num_widths
        half_L = 0.5 * self.min_distance

        print(
            f"[SweepLineCpp] {n_north} coils x {num_widths} widths = {n_vars} vars; "
            f"building z-events...",
            flush=True,
        )

        # Allocate C arrays for effective intervals (indexed by encoded = n*NW+s)
        z_lo_arr = <double*>malloc(n_vars * sizeof(double))
        z_hi_arr = <double*>malloc(n_vars * sizeof(double))
        r_lo_arr = <double*>malloc(n_vars * sizeof(double))
        r_hi_arr = <double*>malloc(n_vars * sizeof(double))

        if not z_lo_arr or not z_hi_arr or not r_lo_arr or not r_hi_arr:
            free(z_lo_arr)
            free(z_hi_arr)
            free(r_lo_arr)
            free(r_hi_arr)
            raise MemoryError("Failed to allocate interval arrays")

        n_intermediate_filters = 0
        _last_print_raw = 0
        try:
            # Precompute effective intervals and build z-events
            for n in range(n_north):
                z_n = pos[n, 2]
                r_n = radii_mv[n]
                for s in range(num_widths):
                    h_s = heights_mv[s]
                    w_s = widths_mv[s]
                    encoded = n * num_widths + s

                    z_lo_arr[encoded] = z_n - h_s * 0.5 - half_L
                    z_hi_arr[encoded] = z_n + h_s * 0.5 + half_L
                    r_lo_arr[encoded] = r_n - w_s * 0.5 - half_L
                    r_hi_arr[encoded] = r_n + w_s * 0.5 + half_L

                    ev.first = z_lo_arr[encoded]
                    ev.second.first = 1   # START
                    ev.second.second = encoded
                    z_events.push_back(ev)

                    ev.first = z_hi_arr[encoded]
                    ev.second.first = 0   # END
                    ev.second.second = encoded
                    z_events.push_back(ev)

            # Sort events: (coord, etype) — END(0) before START(1) at same coord
            sort(z_events.begin(), z_events.end())

            num_events = z_events.size()
            print(
                f"[SweepLineCpp] {num_events} z-events built; starting plane sweep...",
                flush=True,
            )

            # Plane sweep (z-axis)
            active.clear()

            for i in range(num_events):
                ev = z_events[i]
                if ev.second.first == 1:  # START
                    active.insert(ev.second.second)
                else:                      # END
                    active.erase(ev.second.second)

                # Emit 1D cliques when there is a gap to the next event
                if (active.size() > 0
                        and i + 1 < num_events
                        and z_events[i + 1].first > ev.first):
                    _solve_1d_cliques(active, r_lo_arr, r_hi_arr, raw_cliques)

                    n_raw = <int>raw_cliques.size()

                    # Periodic progress print every 100k raw cliques
                    if n_raw - _last_print_raw >= 100000:
                        _last_print_raw = n_raw
                        print(
                            f"[SweepLineCpp] sweep event {i}/{num_events}, "
                            f"active={active.size()}, raw_cliques={n_raw}",
                            flush=True,
                        )

                    # Flush: when cap reached, filter and re-insert survivors, then continue.
                    # _filter_subsets_cpp clears raw_cliques internally (after copying to a
                    # vector) so there is never a moment when both the set and the full copy
                    # exist simultaneously beyond the initial extraction loop.
                    if n_raw >= max_raw_cliques:
                        n_intermediate_filters += 1
                        print(
                            f"[SweepLineCpp] Flush #{n_intermediate_filters}: {n_raw} raw cliques "
                            f"at event {i}/{num_events}. Running intermediate filter...",
                            flush=True,
                        )
                        t_flush = time.perf_counter()

                        # Pure C++ filter: raw_cliques is cleared inside the call,
                        # survivors come back as C++ vectors — zero Python objects created.
                        survivors = _filter_subsets_cpp(raw_cliques)
                        survivor_count = <int>survivors.size()

                        # Re-insert survivors directly (they are already sorted C++ vectors).
                        for j in range(survivor_count):
                            raw_cliques.insert(survivors[j])

                        _last_print_raw = survivor_count
                        print(
                            f"[SweepLineCpp] Flush #{n_intermediate_filters} done in "
                            f"{time.perf_counter()-t_flush:.2f}s: "
                            f"{n_raw} -> {survivor_count} cliques. Continuing sweep...",
                            flush=True,
                        )

        finally:
            free(z_lo_arr)
            free(z_hi_arr)
            free(r_lo_arr)
            free(r_hi_arr)

        t_sweep = time.perf_counter()
        elapsed_sweep = t_sweep - t_start
        n_raw = <int>raw_cliques.size()

        print(
            f"[SweepLineCpp] Sweep done in {elapsed_sweep:.2f}s, "
            f"{n_raw} raw cliques "
            f"({'after ' + str(n_intermediate_filters) + ' flush(es)' if n_intermediate_filters else 'no flushes'}). "
            f"Starting final subset filter...",
            flush=True,
        )

        # Final filter: pure C++ (raw_cliques is cleared inside the call).
        survivors = _filter_subsets_cpp(raw_cliques)

        # Decode to Python frozensets exactly once, only at the very end.
        maximal = []
        for j in range(<int>survivors.size()):
            maximal.append(_clique_to_frozenset(survivors[j], num_widths))

        # Free the C++ survivor vectors immediately; maximal (Python) is the owner now.
        survivors.clear()

        elapsed = time.perf_counter() - t_start
        elapsed_filter = elapsed - elapsed_sweep
        total_members = sum(len(c) for c in maximal)

        self.last_run_stats = SweepLineStats(
            num_raw_cliques=n_raw,
            num_maximal_cliques=len(maximal),
            total_members=total_members,
            elapsed_sweep_s=elapsed_sweep,
            elapsed_filter_s=elapsed_filter,
            elapsed_total_s=elapsed,
        )

        print(
            f"[SweepLineCpp] Elapsed: {elapsed:.4f}s "
            f"(sweep: {elapsed_sweep:.4f}s, filter: {elapsed_filter:.4f}s) | "
            f"Raw cliques before filter: {n_raw} | "
            f"Maximal cliques: {len(maximal)} | "
            f"Clique table members: {total_members}",
            flush=True,
        )

        return maximal

In [30]:
import gurobipy as gp
from gurobipy import GRB

def solve_improved_model(
    coil_positions, coil_radii, coil_widths, coil_heights, 
    coil_cross_section_areas, amns_matrix
):
    """
    Solves the MRI magnet design problem using the Sweep-line Maximal Clique 
    alternative formulation.
    """
    # 1. Enumerate Maximal Cliques
    enumerator = SweepLineCliqueEnumeratorCpp(
        positions=coil_positions,
        radii=coil_radii,
        widths=coil_widths,
        heights=coil_heights,
        min_distance=COIL_MIN_DISTANCE
    )
    
    # We only need cliques for the north pole due to the symmetry constraint
    cliques_list = enumerator.enumerate_cliques(north_only=True)
    
    # 2. Initialize Gurobi Model
    model_alt = gp.Model("improved_clique_optimization")
    model_alt.setParam('FeasibilityTol', 1e-8)
    model_alt.setParam('MIPGap', 0.01)
    model_alt.setParam('IntegralityFocus', 1)
    model_alt.setParam('Threads', 1)
    model_alt.setParam('Seed', 42)

    num_coils = len(coil_positions)
    num_widths = len(coil_widths)
    num_opt_points = amns_matrix.num_opt_points  # derived from AMNS, not global VOI_NUM_POINTS

    # Variables
    vars_i = model_alt.addVars(num_coils, num_widths, lb=-GRB.INFINITY, name="i")
    vars_x = model_alt.addVars(num_coils, num_widths, vtype=GRB.BINARY, name="x")

    # Objective [Eqn 1]
    OBJECTIVE_SCALE = 1e4
    obj = gp.LinExpr(
        (OBJECTIVE_SCALE * coil_radii[:, None] * coil_cross_section_areas[None, :]).flatten().tolist(),
        list(vars_x.values())
    )
    model_alt.setObjective(obj, GRB.MINIMIZE)

    # 3. Add Constraints
    # Homogeneity [Eqn 2 & 3]
    if amns_matrix.Bz.ndim == 3:
        bz_2d = amns_matrix.Bz.reshape(num_opt_points, num_coils * num_widths)
    else:
        bz_2d = amns_matrix.Bz
        
    i_flat = [vars_i[n, s] for n in range(num_coils) for s in range(num_widths)]
    upper_b, lower_b = B_0 * (1 + EPSILON), B_0 * (1 - EPSILON)

    for m in range(num_opt_points):
        expr = gp.LinExpr(bz_2d[m].tolist(), i_flat)
        model_alt.addConstr(expr >= lower_b)
        model_alt.addConstr(expr <= upper_b)

    # Current Density [Eqn 4]
    for n in range(num_coils):
        for s in range(num_widths):
            i_max = J_C * coil_cross_section_areas[s] / COIL_TURNS
            model_alt.addConstr(vars_i[n, s] <= i_max * vars_x[n, s])
            model_alt.addConstr(vars_i[n, s] >= -i_max * vars_x[n, s])

    # Symmetry (North/South)
    half = num_coils // 2
    for n in range(half):
        for s in range(num_widths):
            model_alt.addConstr(vars_x[n, s] == vars_x[n + half, s])

    # 4. Apply Maximal Clique Constraints [Eqn 6a]
    # This replaces the pairwise overlap and unique width constraints
    for i, clique in enumerate(cliques_list):
        model_alt.addConstr(gp.quicksum(vars_x[n, s] for n, s in clique) <= 1)

    # 5. Optimize
    model_alt.update()
    start_time = time.time()
    model_alt.optimize()
    
    print(f"Solve time: {time.time() - start_time:.2f}s")
    print(f"Objective: {model_alt.ObjVal if model_alt.SolCount > 0 else 'N/A'}")

### 4.3 Experiment with a Smaller Instance

In [31]:
coil_widths, coil_heights, coil_cross_section_areas, coil_positions, coil_radii, _ = generate_coil_domain(
    coil_grid_spacing=0.02, generate_overlap_pairs=False
)
AMNS_MATRIX_40_200_20 = load_amns("https://github.com/Vjameli/magnet-design-MIE1603/raw/refs/heads/main/AMNS_40_200_20.npz")

print("Number of optimizing points:", VOI_NUM_POINTS)
print("Number of coil configurations:", len(coil_positions))
print("Number of cross-sectional area options:", COIL_NUM_WIDTHS)

Number of optimizing points: 40
Number of coil configurations: 200
Number of cross-sectional area options: 20


In [32]:
# Run improved model
solve_improved_model(
    coil_positions, coil_radii, coil_widths, 
    coil_heights, coil_cross_section_areas, AMNS_MATRIX_40_200_20
)

[SweepLineCpp] enumerate_cliques: north_only=True, max_raw_cliques=1000000
[SweepLineCpp] 100 coils x 20 widths = 2000 vars; building z-events...
[SweepLineCpp] 4000 z-events built; starting plane sweep...
[SweepLineCpp] sweep event 2779/4000, active=900, raw_cliques=100113
[SweepLineCpp] Sweep done in 0.17s, 131494 raw cliques (no flushes). Starting final subset filter...
[SweepLineCpp] Elapsed: 0.5894s (sweep: 0.1674s, filter: 0.4220s) | Raw cliques before filter: 131494 | Maximal cliques: 7300 | Clique table members: 1247709
Set parameter FeasibilityTol to value 1e-08
Set parameter MIPGap to value 0.01
Set parameter IntegralityFocus to value 1
Set parameter Threads to value 1
Set parameter Seed to value 42
Gurobi Optimizer version 13.0.0 build v13.0.0rc1 (mac64[arm] - Darwin 24.4.0 24E263)

CPU model: Apple M1
Thread count: 8 physical cores, 8 logical processors, using up to 1 threads

Non-default parameters:
FeasibilityTol  1e-08
MIPGap  0.01
IntegralityFocus  1
Seed  42
Threads  1

### 4.4 Experiment with a Larger Instance

In [33]:
coil_widths, coil_heights, coil_cross_section_areas, coil_positions, coil_radii, _ = generate_coil_domain(
    coil_grid_spacing=0.01, generate_overlap_pairs=False
)

# Use the 395-point AMNS (80 boundary + 315 interior) — already loaded above
print("Number of optimizing points:", AMNS_MATRIX_395_702_20.num_opt_points)
print("Number of coil configurations:", len(coil_positions))
print("Number of cross-sectional area options:", COIL_NUM_WIDTHS)

Number of optimizing points: 395
Number of coil configurations: 702
Number of cross-sectional area options: 20


In [34]:
# Run improved model with 395-point AMNS (80 boundary + 315 interior)
solve_improved_model(
    coil_positions, coil_radii, coil_widths, 
    coil_heights, coil_cross_section_areas, AMNS_MATRIX_395_702_20
)

[SweepLineCpp] enumerate_cliques: north_only=True, max_raw_cliques=1000000
[SweepLineCpp] 351 coils x 20 widths = 7020 vars; building z-events...
[SweepLineCpp] 14040 z-events built; starting plane sweep...
[SweepLineCpp] sweep event 2846/14040, active=2769, raw_cliques=101467
[SweepLineCpp] sweep event 4172/14040, active=3393, raw_cliques=203610
[SweepLineCpp] sweep event 5342/14040, active=3627, raw_cliques=305498
[SweepLineCpp] sweep event 6512/14040, active=3705, raw_cliques=408385
[SweepLineCpp] sweep event 7682/14040, active=3705, raw_cliques=511110
[SweepLineCpp] sweep event 8852/14040, active=3627, raw_cliques=613572
[SweepLineCpp] sweep event 10022/14040, active=3315, raw_cliques=714125
[SweepLineCpp] sweep event 11387/14040, active=2652, raw_cliques=815721
[SweepLineCpp] Sweep done in 3.57s, 901275 raw cliques (no flushes). Starting final subset filter...
[SweepLineCpp] Elapsed: 13.1826s (sweep: 3.5669s, filter: 9.6156s) | Raw cliques before filter: 901275 | Maximal cliques: 

## 5. Comparison of Experimental Results 

<table>
  <thead>
    <tr>
      <th></th>
      <th colspan="2">Small Instance</th>
      <th colspan="2">Large Instance</th>
    </tr>
    <tr>
        <th></th>
        <th>Baseline</th>
        <th>Improved</th>
        <th>Baseline <sup>1</sup></th>
        <th>Improved</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <td>No. of coil configurations</td>
      <td colspan="2">200</td>
      <td colspan="2">702</td>
    </tr>
    <tr>
      <td>No. of optimization points</td>
      <td colspan="2">40</td>
      <td colspan="2">395</td>
    </tr>
    <tr>
      <td>No. of constraints</td>
      <td>810,772</td>
      <td>17,380</td>
      <td>10,908,570</td>
      <td>48,108</td>
    </tr>
    <tr>
      <td>Constraints pre-compute time [sec] <sup>2</sup></td>
      <td>1.73</td>
      <td>0.59</td>
      <td>20.22</td>
      <td>13.18</td>
    </tr>
    <tr>
      <td>Gurobi solve time [sec]</td>
      <td>16.42</td>
      <td>23.87</td>
      <td> -  </td>
      <td>307.89</td>
    </tr>
    <tr>
      <td>Total time [sec] <sup>3</sup></td>
      <td>18.15</td>
      <td>24.46</td>
      <td> - </td>
      <td>321.07</td>
    </tr>
    <tr>
      <td>Gurobi simplex iterations</td>
      <td>42,347</td>
      <td>54,006</td>
      <td> - </td>
      <td>106,871</td>
    </tr>
    <tr>
      <td>Gurobi nodes explored</td>
      <td>39</td>
      <td>345</td>
      <td>1</td>
      <td>91</td>
    </tr>
  </tbody>
  <tfoot>
    <tr>
        <td colspan="5">
        Notes: <br />
        <sup>1</sup> The large instance baseline is numerically unstable — the LP relaxation at the root node diverges. <br />
        <sup>2</sup> Pre-computed constraints refer to the overlapping constraints for the baseline models and the maximal cliques for the improved models. <br />
        <sup>3</sup> Total time = constraints pre-compute time + Gurobi solve time.
        </td>
    </tr>
  </tfoot>
</table>

The results show that the alternative formulation significantly reduces the number of constraints, and consequently memory usage. These efficiencies not only increase the maximum solvable problem size but also enhance the numerical stability of the model.

## 6. Discussion and Propositions

This report concludes with several propositions that provide the mathematical foundation and supporting arguments for the proposed formulations. For the sake of completeness and exposition, formal proofs are included; however, we do not claim these as original results. In particular, Propositions 3 and 4 come from trivial results from graph theory, but are presented here to provide a self-contained report.

### 6.1 Baseline Overlapping Constraints

In the preceding discussion, we focus on the overlapping constraints defined by Eq (6). However, because the proposed formulation replaces both Eq (5) and Eq (6), it is necessary to examine the specific role Eq (5) serves within the baseline model.

For $n_i \in \mathcal{N}$ , all possible coil instances $ (n_i, s), \forall s \in \mathcal{S}$ share the same center and therefore overlap with each other. Consequently, these overlaps would be captured by constraints (6) if allowing $n=n'$. Proposition 1 below shows that this is indeed the case when the non-overlapping constraints are defined in a pairwise fashion. 

>**Proposition 1.** 

Let $\mathbf{x} \in [0,1]^{|\mathcal{N}| \times |\mathcal{S}|}$ and $\mathcal{C}_{\text{conflict}}$ denote the set of all pairs of variables $((n,s), (n',s'))$ that overlap, including cross-sections at the same location ($n = n', s \neq s'$). The continous relaxations of the two different formulations are defined as
\begin{equation}
    P_1 = \{ \mathbf{x} \in [0,1]^{|\mathcal{N}| \times |\mathcal{S}|} \mid x_n^s + x_{n'}^{s'} \le 1, \quad \forall ((n,s), (n',s')) \in \mathcal{C}_{\text{conflict}} \}\text{, and} \tag{9}
\end{equation}

\begin{equation}
    P_2 = \left\{ \mathbf{x} \in [0,1]^{|\mathcal{N}| \times |\mathcal{S}|} \;\middle|\; 
    \begin{array}{l} 
        x_n^s + x_{n'}^{s'} \le 1, \quad \forall n \neq n', (s,s') \in \mathcal{S}_{nn'} \\ 
        \sum_{s \in \mathcal{S}} x_n^s \le 1, \quad \forall n \in \mathcal{N} 
    \end{array} 
    \right\} \tag{10}
\end{equation}

Then $P_2 \subset P_1$.


> **Proof** 

First, we show $P_2 \subseteq P_1$. Let $\mathbf{x} \in P_2$. For any $n \neq n'$, Eq (6) is explicitly satisfied in $P_2$. For any $n = n'$ but $s \neq s'$, we have $x_n^s + x_n^{s'} \le \sum_{k \in \mathcal{S}} x_n^k \le 1$. Thus,  $P_2 \subseteq P_1$.

Now we show $P_2 \subset P_1$. Pick an $n \in \mathcal{N}$ that has at least three available cross-sections (e.g., $A, B$, and $C$). Construct a fractional solution $\mathbf{x}^*$ where $x_n^A = x_n^B = x_n^C = 0.5$, and all other variables in $\mathbf{x}^*$ are set to $0$. Evaluating $\mathbf{x}^*$ in $P_1$ for this $n$ yields:
\begin{align*}
    x_n^A + x_n^B &= 0.5 + 0.5 = 1 \le 1 \\
    x_n^A + x_n^C &= 0.5 + 0.5 = 1 \le 1 \\
    x_n^B + x_n^C &= 0.5 + 0.5 = 1 \le 1
\end{align*}
Since all other variables are $0$, $\mathbf{x}^*$ satisfies all constraints in $P_1$, so $\mathbf{x}^* \in P_1$. 
However, evaluating constraint Eq (5) for $\mathbf{x}^*$ in $P_2$ yields:
\begin{equation*}
    x_n^A + x_n^B + x_n^C = 0.5 + 0.5 + 0.5 = 1.5 \not\le 1
\end{equation*}
Thus, $\mathbf{x}^* \notin P_2$, so $P_2 \subset P_1$.

>**End of Proof**

That being said, one might attemp to further decompose the variable index $n$ into distinct spatial coordinates $z$ and radii $r$,
creating a variable $x_{z,r}^s$. This would allow for a constraint over the radius index:
\begin{equation*}
    \sum_{r} \sum_{s} x_{z,r}^s \le 1, \quad \forall z
\end{equation*}
Despite this formulation being tighter, it removes a subset of feasible solutions. Coils can occupy the same axial position $z$ acting as concentric coils, but the formulation above would force the solver to select at most one coil per axial plane. Therefore, we keep the index $n$, which is preprocessed to represent a specific $(z, r)$ instance.

### 6.2 Single Cross Section Constraint and Maximal Cliques

One might also consider whether Eq (5) remains necessary when maximal cliques are employed. Proposition 2 demonstrates that this constraint is indeed redundant; it represents a sub-clique that is inherently subsumed by the more general formulation in Eq (6a).

> **Proposition 2.**

Let $\mathcal{C} \in \mathbb{C}$ be the set of all maximal cliques in $G$. If the model enforces the maximal clique inequalities 
\begin{equation*}
    \sum_{(n,s) \in \mathcal{C}} x_n^s \leq 1, \quad \forall \mathcal{C} \in \mathbb{C}
\end{equation*}
then the single cross-section constraints $\sum_{s \in \mathcal{S}} x_n^s \le 1, \forall n \in \mathcal{N}$ are redundant.

> **Proof**

Let $V_n = \{(n,s) \mid s \in \mathcal{S}\}$ be the subset of vertices in $G$ corresponding to all available cross-sections at location $n$. Since the available cross-sections at location $n$ are concentric, by the Helly property, they all share the same center and, therefore, overlap. Since they overlap, an edge exists between them in $E$, and $V_n$ induces a clique in the conflict graph $G$.
By definition, any clique in a graph is either maximal itself or is a proper subset of at least one maximal clique. Thus, there exists some maximal clique $\mathcal{C}^* \in \mathbb{C}$ such that $V_n \subseteq \mathcal{C}^*$.

By construction, our new formulation includes the maximal clique inequality for $\mathcal{C}^*$:
\begin{equation*}
    \sum_{(i,j) \in \mathcal{C}^*} x_i^j \leq 1 \tag{6a}
\end{equation*}

Since $V_n \subseteq \mathcal{C}^*$ and $x_i^j \ge 0$ for all $(i,j)$, we can decompose the sum:
\begin{equation*}
    \sum_{s \in \mathcal{S}} x_n^s = \sum_{(n,s) \in V_n} x_n^s \leq \sum_{(i,j) \in \mathcal{C}^*} x_i^j \leq 1
\end{equation*}

Thus, constraint $\sum_{s \in \mathcal{S}} x_n^s \leq 1$ is implied by the maximal clique inequality for $\mathcal{C}^*$ and is therefore redundant. 

>**End of Proof**

Note that if the single cross-section constraint Eq(5) is employeed in addition to the maximal cliques, the LP relaxation optimal solution will be the same.
Consequently, the new formulation replaces constraints defined by Eq(5) and Eq(6) with all the maximal cliques defined by Eq(6a).


### 6.3 Clique Cover

An interesting observation about set-packing constraint is that even though the set-packing constraints are essentially summation over vertices, covering all vertices is not sufficient to avoid overlap. One can easily prove that by considering the example below.

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/simple_example.png?raw=true" width="30%">
    <p>Figure 5: Conflict Graph Example with 5 Vertices<p>
</center>

If the goal is to cover all the vertices, $x_1 + x_2 + x_3 \leq 1$ and $x_4 + x_5 \leq 1$ would suffice. However, this is clearly insufficient as overlaps between coil 2 and coil 4 or 5 are not captured. By adding the term $x_2$ in second constraint, $x_2 + x_4 + x_5 \leq 1$, we then have two cliques that cover all the edges thus covering all the possible overlaps.

However, instead of focusing on vertices, graph theory also defines the **edge clique cover**: a collection of cliques that covers all edges in a graph. The minimum cardinality of such a collection is known as the **edge clique cover number**. Naturally, this raises the question of whether an edge clique cover alone is sufficient to define the non-overlapping logic.

> **Proposition 3.**

Let $X \subseteq \{0,1\}^{|V|}$ be the set of all non-overlapping coil configurations. If we model $X$ using a system of $m$ linear packing constraints of the form

\begin{equation*}
    \sum_{v \in S_k} x_v \le 1, \quad k = 1, \dots, m
\end{equation*}

where $S_k \subseteq V$, then the minimum number of constraints $m^*$ required to describe $X$ is exactly the edge clique cover number of the intersection graph.

> **Proof**

Let $\mathcal{F} = \{S_1, S_2, \dots, S_m\}$. The feasible region defined by $\mathcal{F}$ is:
\begin{equation*}
    X_{\mathcal{F}} = \left\{ \mathbf{x} \in \{0,1\}^{|V|} \;\middle|\; \sum_{v \in S_k} x_v \le 1, \quad \forall S_k \in \mathcal{F} \right\}
\end{equation*}
*We require $X_{\mathcal{F}} = X$. 
Suppose for contradiction that $\exists \, \, S_k \in \mathcal{F}$ that is not a clique. Then $\exists \, \, u, w \in S_k \mid \{u,w\} \notin E$. Thus, define $\mathbf{\hat{x}}$ where $\hat{x}_u = 1$, $\hat{x}_w = 1$, and $\hat{x}_v = 0$ for all other $v \in S_k$. By construction, $\mathbf{\hat{x}} \in X$.*

However, evaluating $\mathbf{\hat{x}}$ in the constraint for $S_k$ yields:
\begin{equation*}
    \sum_{v \in S_k} \hat{x}_v = \hat{x}_u + \hat{x}_w = 2 \not\le 1
\end{equation*}
This implies $\mathbf{\hat{x}} \notin X_{\mathcal{F}}$, contradicting $X_{\mathcal{F}} = X$. Thus, every $S_k \in \mathcal{F}$ must be a clique.

Now, suppose for contradiction that $\mathcal{F}$ is not an edge clique cover. Conversely, $\exists \,\, \{u,w\} \in E \mid \{u,w\} \not\subset S_k \in \mathcal{F}, \forall k$. This means that for every constraint $k$, $|S_k \cap \{u,w\}| \le 1$.
Consider the same $\mathbf{\hat{x}}$ defined before. Because $\{u,w\} \in E$, $\mathbf{\hat{x}} \notin X$. 
However, evaluating $\mathbf{\hat{x}}$ against any constraint $S_k \in \mathcal{F}$ yields:
\begin{equation*}
    \sum_{v \in S_k} \hat{x}_v = |S_k \cap \{u,w\}| \le 1
\end{equation*}
Thus, $\mathbf{\hat{x}} \in X_{\mathcal{F}}$, which again contradicts $X_{\mathcal{F}} = X$. Therefore, $\mathcal{F}$ must cover every edge in $E$.

Since every valid formulation of this form is an edge clique cover, the formulation with the absolute smallest number of constraints $m^*$ corresponds to the edge clique cover number. 

>**End of Proof**


Consider the following example, where the size of the edge clique cover set is smaller than the set of all maximal cliques, provides a illustrative example for Proposition 3.

<center>
    <img src="https://github.com/Vjameli/magnet-design-MIE1603/blob/main/images/edge_clique_cover.png?raw=true" width="75%">
    <p>Figure 6: Modelling example rectangle overlap problem with conflict graph and maximal clique<p>
</center>

The rectangular overlaps define the cliques:
\begin{align*}
&\mathcal{C}_1 = x_1 + x_2 + x_4 \\
&\mathcal{C}_2 = x_2 + x_3 + x_5 \\
&\mathcal{C}_3 = x_1 + x_3 + x_6 \\
&\mathcal{C}_4 = x_1 + x_2 + x_3. \\
\end{align*}

All cliques are valid inequalities, but only the first three are necessary to cover all the edges.

While the edge clique cover number is the minimum number of constraints required to enforce the non-overlapping logic, minimizing the constraint count is not necessarily the best strategy. The following proposition demonstrates that the constraint set defined by all maximal cliques provides a tighter linear relaxation than a formulation derived solely from a minimal edge clique cover.
> **Proposition 4**

Let $\mathcal{F}^*$ be a minimum edge clique cover of $G$. Define the continuous linear programming relaxations associated with these two sets of constraints as:
\begin{align}
    P_3 &= \left\{ \mathbf{x} \in [0,1]^{|V|} \;\middle|\; \sum_{v \in \mathcal{C}} x_v \le 1, \quad \forall \mathcal{C} \in \mathbb{C} \right\} \\
    P_4 &= \left\{ \mathbf{x} \in [0,1]^{|V|} \;\middle|\; \sum_{v \in S} x_v \le 1, \quad \forall S \in \mathcal{F}^* \right\}
\end{align}
Then $P_3 \subset P_4$.

> **Proof**

First, we show $P_3 \subseteq P_4$. Any clique $S \in \mathcal{F}^*$ is either a maximal clique itself ($S \in \mathbb{C}$) or a proper subset of some maximal clique $\mathcal{C}' \in \mathbb{C}$. If $S \in \mathbb{C}$, the constraint is identical in both formulations. If $S \subset \mathcal{C}'$, then  $\forall \,\mathbf{x} \ge \mathbf{0}$, $\sum_{v \in S} x_v \le \sum_{v \in \mathcal{C}'} x_v \le 1$. Thus, if $\mathbf{x} \in P_3$, then $\mathbf{x} \in P_4$ $\Rightarrow$ $P_3 \subseteq P_4$.

To prove $P_3 \subset P_4$, consider a conflict graph $G=(V,E)$ with six vertices $V = \{1, 2, 3, 4, 5, 6\}$. Let the graph contain four maximal cliques:
\begin{equation*}
    \mathbb{C} = \left\{ \{1,2,4\}, \{2,3,5\}, \{1,3,6\}, \{1,2,3\} \right\}
\end{equation*}
The edges of $G$ are covered by the first three cliques. Therefore, a valid minimal edge clique cover is $\mathcal{F}^* = \left\{ \{1,2,4\}, \{2,3,5\}, \{1,3,6\} \right\}$, and the fourth clique $\{1,2,3\}$ is omitted.
Consider the fractional point $\mathbf{x}^* = (0.5, 0.5, 0.5, 0, 0, 0)$. 
Evaluating $\mathbf{x}^*$ in $P_4$ yields:
\begin{align*}
    x_1 + x_2 + x_4 &= 0.5 + 0.5 + 0 = 1 \le 1 \\
    x_2 + x_3 + x_5 &= 0.5 + 0.5 + 0 = 1 \le 1 \\
    x_1 + x_3 + x_6 &= 0.5 + 0.5 + 0 = 1 \le 1
\end{align*}
Thus, $\mathbf{x}^* \in P_4$.
However, evaluating $\mathbf{x}^*$ against the omitted maximal clique constraint in $P_3$ yields:
\begin{equation*}
    x_1 + x_2 + x_3 = 0.5 + 0.5 + 0.5 = 1.5 > 1
\end{equation*}
So $\mathbf{x}^* \notin P_3$. Therefore $P_3 \neq P_4$ $\Rightarrow$ $P_3 \subset P_4$.


## 7. References

1. G. P. Liney, B. Whelan, B. Oborn, M. Barton, and P. Keall, “MRI-Linear Accelerator Radiotherapy Systems,” Clinical Oncology, vol. 30, no. 11, pp. 686–691, Nov. 2018, doi: 10.1016/j.clon.2018.08.003.
1. A. Sattarov, P. McIntyre, and L. Motowidlo, “High-Field Open MRI for Breast Cancer Screening,” IEEE Trans. Appl. Supercond., vol. 25, no. 3, pp. 1–5, Jun. 2015, doi: 10.1109/TASC.2014.2377049.
1. Hao Xu, S. M. Conolly, G. C. Scott, and A. Macovski, “Homogeneous magnet design using linear programming,” IEEE Trans. Magn., vol. 36, no. 2, pp. 476–483, Mar. 2000, doi: 10.1109/20.825817.
1. I. Dayarian, T. C. Y. Chan, D. Jaffray, and T. Stanescu, “A mixed-integer optimization approach for homogeneous magnet design,” Technology, vol. 06, no. 02, pp. 49–58, Jun. 2018, doi: 10.1142/S2339547818500036.
1. L. K. Forbes, S. Crozier, and D. M. Doddrell, “Rapid computation of static fields produced by thick circular solenoids,” IEEE Trans. Magn., vol. 33, no. 5, pp. 4405–4410, Sep. 1997, doi: 10.1109/20.620453.
1. M. O. Rodrigues and F. M. B. Toledo, "A Clique Covering MIP Model for the Irregular Strip Packing Problem," Comput. Oper. Res., vol. 87, pp. 221–234, 2017.
1. Manfred W Padberg. On the facial structure of set packing polyhedra. Mathematical programming, 5(1):
199–215, 1973
1. H. Imai and T. Asano, "Finding the Connected Components and a Maximum Clique of an Intersection Graph of Rectangles in the Plane," J. Algorithms, vol. 4, no. 4, pp. 310–323, 1983.
1. Bram Verweij and Karen Aardal. An optimisation algorithm for maximum independent set with applications
in map labelling. Technical report, Utrecht University, 1999
1. Vernon Ning Hsu, Dilip Chhajed, and Timothy J Lowe. Tool design problems in a punch press flexible
manufacturing system. IIE Transactions, 30(4):331–340, 1998.
1. Bernardo Martin-Iradi, Dario Pacino, and Stefan Ropke. The multi-port berth allocation problem with speed
optimization: Exact methods and a cooperative game analysis. European Journal of Operational Research, 2020.